# Universal Stock LSTM

Trains a single LSTM on multiple symbols using scale-free features (log returns, ratios).
No scaler required — features are dimensionless and comparable across stocks.

**Flow:**
1. Load symbols from cache
2. Compute scale-free features + chronological train/test split per stock
3. Pool all stocks, build DataLoaders
4. Train `StockLSTM(input_size=7)`
5. Evaluate directional accuracy
6. Live inference on a single stock

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from sentiment.log import setup_logging
from sentiment.sources.cache import MarketDataCache
import priceEstimation.sources.dataloader as dl
import priceEstimation.sources.models as md
import priceEstimation.sources.train_evaluate as tr

setup_logging()

## Config

In [ ]:
YEAR = 2025
SEQ_LENGTH = 20
BATCH_SIZE = 32
N_EPOCHS = 50
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

In [ ]:
symbols = [
    "AAPL", "MSFT", "NVDA", "AMZN", "GOOGL",
    "META", "TSLA", "AVGO", "PEP", "COST",
]

cache = MarketDataCache()

## Load & preprocess all symbols

In [ ]:
X_train, y_train, X_test, y_test = dl.load_all_symbols(
    cache, symbols, YEAR, seq_length=SEQ_LENGTH
)

print(f"X_train: {X_train.shape}  y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}   y_test:  {y_test.shape}")

## DataLoaders

In [ ]:
train_loader = dl.get_stock_loader(X_train, y_train, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = dl.get_stock_loader(X_test,  y_test,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}  |  Test batches: {len(test_loader)}")

## Model

In [ ]:
model = md.StockLSTM(input_size=7, hidden_size=64, num_layers=2).to(DEVICE)
print(model)

## Training

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

train_log, test_log = tr.train_cycle(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    train_loader=train_loader,
    test_loader=test_loader,
    n_epochs=N_EPOCHS,
    device=DEVICE,
)

## Evaluation

In [ ]:
dir_acc = tr.directional_accuracy(model, test_loader, device=DEVICE)
print(f"Directional accuracy: {dir_acc:.3f}  (chance = 0.500)")

In [ ]:
preds, actuals = tr.plot_prediction(model, test_loader, device=DEVICE)

## Save model

In [ ]:
torch.save(model.state_dict(), "universal_model.pt")
print("Saved to universal_model.pt")

## Live inference

Given any single stock in the cache, predict the next bar's log return
and convert to an expected price.

In [ ]:
import numpy as np

infer_symbol = "AAPL"

df_infer = dl.load_data(cache, infer_symbol, YEAR)
X_infer, _ = dl.preprocess_data(df_infer, seq_length=SEQ_LENGTH)

last_window = torch.tensor(X_infer[-1:], dtype=torch.float32).to(DEVICE)

model.eval()
with torch.no_grad():
    pred_log_return = model(last_window).item()

last_close = df_infer["close"].iloc[-1]
predicted_price = last_close * np.exp(pred_log_return)

print(f"{infer_symbol} last close : ${last_close:.2f}")
print(f"Predicted log return : {pred_log_return:+.5f}")
print(f"Predicted next close : ${predicted_price:.2f}")